A fasterai tutorial is a claim about what the library does to a real model. This page is
the contract those claims are held to, and it is also the enforcement: the hidden cells
below parse every tutorial in `nbs/tutorials/` (plus `quickstart` and `overview`) and
assert the rules that follow. They run in the default test suite, so a page that breaks
the contract turns CI red before it reaches the site.

## The voice

A tutorial reports what one run measured. It does not rank, advise or promise.

| Instead of | Write |
|---|---|
| "the best criteria for this" | "in this run, `large_final` ended at 92.69% (1370/1478)" |
| "always fold batch norm before export" | "folding rewrites the convolution weights; this page folds before exporting" |
| "up to 4x faster" | "633 ms against 26 ms on this box, single run" |
| "minimal accuracy loss" | "-3.1 points, and the two Wilson intervals overlap" |

These words never appear in tutorial prose, because each of them states a preference or a
guarantee that no cell on the page measured: **best**, **fastest**, **always**, **never**,
**recommended**, **rule of thumb**, **key finding**, **tip**, **guaranteed**, **optimal**,
**when to use**, **for free**, **free speedup**, **no drop in accuracy**, **without any
drop**, and the verdict emoji. Code is not prose: `warnings.simplefilter('always')` is
fine, and so is a file named `best.pt`.

## Every number comes from the page

The rule is mechanical: a number stated in prose must be a number a cell on the same page
prints, exactly or as a rounding of it. A percentage may be printed as a proportion
(`0.926928` backs `92.69%`), a parameter count may be printed in full (`134309962` backs
`134 million`). What is *not* allowed is a number nobody computed: a ratio, a delta or a
speed-up that only exists in the sentence. If a page wants to say "3.9x smaller", a cell
prints the ratio.

Confidence bounds are the one exception, and only because they are checked harder: an
interval written next to a `k/n` is recomputed from `k` and `n` and has to match.

## Measurement rules

- **Accuracy carries its n.** A sentence stating an accuracy carries `n = N`, or a `k/n`,
  or names the evaluation set size ("the 1478 validation images").
- **Percentages that come from counts are written with the counts**, and the two must
  agree: `92.69% (1370/1478)`.
- **Intervals are Wilson 95% intervals** and are recomputed from `k/n` — on whichever
  scale the page prints, percentages or proportions.
- **A latency carries a scope line**: "single run", "one run", "this run" or
  "Measured on ...". A number with a `ms`, `µs` or `FPS` unit and no scope is a claim
  about a machine the reader does not have.
- **A sentinel is never a result.** `0.00 MB`, `0 parameters`, `nan ms`, `accuracy: -1`
  are failed measurements; they may be discussed, never shown as an outcome.

## Required elements

Front matter with `title`, `description` and `skip_exec: true`; a `## Summary` section;
a `## See Also` section; every link resolving to a page or an image that exists.
`tutorials/quantize/deployable_export.ipynb` is the reference for the shape of a page.

## The rules the checks enforce

| Rule | What it asserts | Waivable |
|---|---|---|
| R1 | No cell output is an error or contains a traceback | no |
| R2 | A visible cell that produces something shows it, and no output is a bare placeholder | yes |
| R3 | No sentinel value (`0.00 MB`, `0 parameters`, `nan`, `-1`) is shown as a result | no |
| R4 | Every number in prose is printed by a cell on the same page | yes |
| R4b | A page reporting a latency carries a scope line | yes |
| R5 | The prose uses none of the forbidden words | yes |
| R6 | Names shown in prose exist: blocks parse, keyword arguments and vocabularies are real | yes |
| R7a | Front matter, `## Summary` and `## See Also` are present | yes |
| R7b | Every internal link resolves | no |
| R8 | The sidebar and the tutorials folder agree, both directions | no |
| R9 | Accuracy sentences carry n; `k/n` and intervals are recomputed | yes |
| R10 | No cell output is a stderr stream, hidden cells included | yes |

"Waivable" means the rule can hold an entry in the quarantine described below. The four
that cannot be waived are the ones whose failure means the page is broken rather than
imperfect: a failure shown as a result, and a link or sidebar entry that goes nowhere.

## Running the checks

```
nbdev_test --path nbs/tutorial_contract.ipynb          # seconds; this is what CI runs
```

And, on a machine with the GPU, the datasets and the optional dependencies, the execution
gate — it re-runs every tutorial in a copy of its folder and reports what the code no
longer reproduces:

```
python -c "from nbdev.test import nbdev_test; nbdev_test(path='nbs/tutorial_contract.ipynb', flags='tutorials')"
```

It takes about an hour. `FASTERAI_TUTORIALS='tutorials/sparse/*'` narrows it. Nothing is
written under `nbs/`, and no output is committed by it: when a page drifts, the remedy is
to re-run that page in a Jupyter kernel and commit the new outputs together with the prose
they support.

## The quarantine, and how it empties

Pages written before this contract carry known violations. Rather than block them, the
gate holds a `QUARANTINE` dict — a per-page list of the rules that page is still allowed
to fail — and enforces three things about it:

1. **It can only shrink.** A frozen copy, `_LANDED`, was taken when this page landed. A
   page or a rule that is not in `_LANDED` cannot enter `QUARANTINE`, so a new violation
   can never be waived — it must be fixed.
2. **It cannot hold an unwaivable rule.**
3. **Entries expire the moment they are unused.** If a quarantined page stops violating
   its rule, the gate fails with *"now passes R4: delete the entry"*. Rewriting a page and
   deleting its entry are one commit.

`_LANDED` is edited by review only; nothing in the machinery writes to it.

To let a single sentence through R5 — an honest caveat that happens to contain a listed
word — add it verbatim to `LEXICON_ALLOW[page]`. The match is the whole sentence, so the
diff shows exactly what was allowed.

## What these checks cannot do

They read structure, not meaning. They cannot tell whether a sentence's conclusion follows
from the number beside it, whether a comparison is controlled, whether a baseline is
degenerate, or whether a `python` block would actually run. Those stay with the reviewer
of every tutorial change. Two known blind spots are worth naming: `Pruner` and
`PruneCallback` accept `**kwargs`, so no signature can refute a made-up keyword on them
(a denylist covers the ones already seen), and a cell whose last statement is a call is
not required to have output, because a call may legitimately return `None`.

In [ ]:
#| hide
# The page model. Every rule below is a pure function of a notebook dict, so the tests
# in this notebook can build a page in memory instead of writing files.
from __future__ import annotations
import ast, json, math, os, re
from dataclasses import dataclass, asdict
from pathlib import Path
from fastcore.test import *

@dataclass(slots=True)
class Violation:
    page: str    # page path relative to `nbs/`, e.g. 'tutorials/sparse/sparsifier.ipynb'
    rule: str    # rule id: 'R1'..'R10'
    where: str   # cell index as a string, or 'frontmatter' / 'page' / 'sidebar'
    detail: str  # the offending token, link or sentence, verbatim

    def as_dict(self) -> dict[str, str]: return asdict(self)
    def __str__(self) -> str: return f"{self.page:50s} {self.rule:4s} {self.where:11s} {self.detail}"

def nbs_dir(start: Path | None = None) -> Path:
    "The directory that holds `_quarto.yml`, searched upwards from `start`"
    p = (start or Path.cwd()).resolve()
    for cand in (p, *p.parents):
        if (cand/'_quarto.yml').exists(): return cand
    raise RuntimeError(f"no _quarto.yml at or above {p}")

def discover_pages(nbs: Path) -> list[str]:
    "Every rendered tutorial page, as a path relative to `nbs/`, sorted"
    out = [p.relative_to(nbs).as_posix() for p in (nbs/'tutorials').rglob('*.ipynb')
           if '.ipynb_checkpoints' not in p.parts and not p.name.startswith('_')]
    out += [n for n in ('quickstart.ipynb', 'overview.ipynb') if (nbs/n).exists()]
    return sorted(out)

def load_page(nbs: Path, rel: str) -> dict: return json.loads((nbs/rel).read_text())

def src(cell: dict) -> str:
    s = cell.get('source', '')
    return s if isinstance(s, str) else ''.join(s)

def frontmatter(nb: dict) -> dict[str, str]:
    "The front matter of the first cell as a flat dict (empty when the page has none)"
    cells = nb.get('cells') or []
    if not cells or cells[0].get('cell_type') != 'raw': return {}
    text = src(cells[0]).strip()
    if not text.startswith('---'): return {}
    body = text.split('---')[1] if text.count('---') >= 2 else ''
    return {k.strip(): v.strip() for k, _, v in
            (l.partition(':') for l in body.splitlines() if ':' in l)}

def markdown_cells(nb: dict) -> list[tuple[int, str]]:
    return [(i, src(c)) for i, c in enumerate(nb.get('cells') or []) if c.get('cell_type') == 'markdown']

def code_cells(nb: dict) -> list[tuple[int, dict]]:
    return [(i, c) for i, c in enumerate(nb.get('cells') or []) if c.get('cell_type') == 'code']

_HIDDEN = re.compile(r'^\s*#\|\s*(hide|include:\s*false)', re.M)

def visible_code_cells(nb: dict) -> list[tuple[int, dict]]:
    "Code cells that reach the rendered page: no `#| hide`, no `#| include: false`"
    return [(i, c) for i, c in code_cells(nb) if not _HIDDEN.search(src(c))]

_FENCE = re.compile(r'^\s*```.*?^\s*```', re.S | re.M)
_SPAN  = re.compile(r'`[^`\n]*`')
_IMG   = re.compile(r'!\[[^\]]*\]\([^)]*\)')
_LINK  = re.compile(r'\[([^\]]*)\]\(([^)\s]*)(?:\s+"[^"]*")?\)')
_HTML  = re.compile(r'<[^>\n]{1,200}>')

def prose(md: str) -> str:
    "What a reader reads: fenced blocks, code spans, link targets and HTML tags removed"
    t = _FENCE.sub('', md)
    t = _IMG.sub('', t)
    t = _LINK.sub(r'\1', t)
    t = _SPAN.sub(' ', t)
    return _HTML.sub(' ', t)

def code_spans(md: str) -> list[str]:
    "Inline `...` spans, back-ticks stripped"
    return [s[1:-1] for s in _SPAN.findall(_FENCE.sub('', md))]

def python_blocks(md: str) -> list[str]:
    "```python fenced blocks"
    return re.findall(r'^\s*```(?:python|py)\s*\n(.*?)^\s*```', md, re.S | re.M)

def table_rows(md: str) -> list[list[str]]:
    "Cells of every `|` table row; the `---` separator row is dropped, row 0 is the header"
    rows = []
    for line in _FENCE.sub('', md).splitlines():
        s = line.strip()
        if not (s.startswith('|') and s.endswith('|')): continue
        cells = [c.strip() for c in s[1:-1].split('|')]
        if all(re.fullmatch(r':?-{2,}:?', c) for c in cells if c): continue
        rows.append(cells)
    return rows

_UNIT_START = re.compile(r'^\s*(?:[-*+]\s|\d+[.)]\s|\||#{1,6}\s|>\s)')

def units(md: str) -> list[str]:
    "Prose split into reading units: a wrapped paragraph is one unit, so is a table row"
    out, cur = [], []
    for line in md.splitlines():
        if not line.strip():
            if cur: out.append(' '.join(cur)); cur = []
            continue
        if _UNIT_START.match(line):
            if cur: out.append(' '.join(cur)); cur = []
            cur = [line.strip()]
            if line.lstrip().startswith('|'): out.append(cur[0]); cur = []
        else: cur.append(line.strip())
    if cur: out.append(' '.join(cur))
    return out

def sentences(md: str) -> list[str]:
    "Units first, then `.!?` splits - so a sentence wrapped over two lines stays whole"
    return [s.strip() for u in units(md) for s in re.split(r'(?<=[.!?])\s+', u) if s.strip()]

_ANSI = re.compile(r'\x1b\[[0-9;]*[A-Za-z]')

def output_text(cell: dict) -> str:
    "Everything an executed cell shows: streams, text/plain, tag-stripped text/html"
    parts = []
    for o in cell.get('outputs') or []:
        if o.get('output_type') == 'stream': parts.append(''.join(o.get('text') or []))
        elif o.get('output_type') == 'error':
            parts.append(f"{o.get('ename', '')}: {o.get('evalue', '')}")
            parts.append('\n'.join(o.get('traceback') or []))
        data = o.get('data') or {}
        for k in ('text/plain', 'text/html'):
            if k in data:
                txt = ''.join(data[k]) if isinstance(data[k], list) else str(data[k])
                parts.append(_HTML.sub(' ', txt) if k == 'text/html' else txt)
    return _ANSI.sub('', '\n'.join(parts))

def page_output_text(nb: dict) -> str: return '\n'.join(output_text(c) for _, c in code_cells(nb))
def page_source_text(nb: dict) -> str: return '\n'.join(src(c) for _, c in visible_code_cells(nb))

In [ ]:
#| hide
# Fixture builder: a page is a dict, so every rule can be tested without touching disk.
def mk_nb(*cells) -> dict: return {'cells': list(cells)}
def raw_cell(source: str) -> dict: return {'cell_type': 'raw', 'metadata': {}, 'source': source}
def md_cell(source: str) -> dict: return {'cell_type': 'markdown', 'metadata': {}, 'source': source}
def code_cell(source: str, *outputs) -> dict:
    return {'cell_type': 'code', 'metadata': {}, 'execution_count': None,
            'source': source, 'outputs': list(outputs)}
def stream(text: str, name: str = 'stdout') -> dict:
    return {'output_type': 'stream', 'name': name, 'text': text}
def result(**data) -> dict:
    keys = {'plain': 'text/plain', 'html': 'text/html', 'png': 'image/png'}
    return {'output_type': 'execute_result', 'execution_count': None, 'metadata': {},
            'data': {keys[k]: v for k, v in data.items()}}
def error(ename: str = 'ValueError', evalue: str = 'boom') -> dict:
    return {'output_type': 'error', 'ename': ename, 'evalue': evalue,
            'traceback': ['Traceback (most recent call last):', f'{ename}: {evalue}']}
def fm(**kw) -> dict:
    return raw_cell('---\n' + ''.join(f'{k}: {v}\n' for k, v in kw.items()) + '---')

_page = mk_nb(fm(title='T', description='D', skip_exec='true'),
              md_cell('A [link](x.html) and a `span`.\n\n```python\nprint(1)\n```\n'),
              code_cell('#| hide\nsecret = 1'),
              code_cell('print(2)', stream('2\n')))
test_eq(frontmatter(_page), {'title': 'T', 'description': 'D', 'skip_exec': 'true'})
test_eq(frontmatter(mk_nb(md_cell('# no front matter'))), {})
test_eq([i for i, _ in markdown_cells(_page)], [1])
test_eq([i for i, _ in code_cells(_page)], [2, 3])
test_eq([i for i, _ in visible_code_cells(_page)], [3])
test_eq(prose(src(_page['cells'][1])).split(), ['A', 'link', 'and', 'a', '.'])
test_eq(code_spans(src(_page['cells'][1])), ['span'])
test_eq(python_blocks(src(_page['cells'][1])), ['print(1)\n'])
test_eq(page_output_text(_page).strip(), '2')

# a paragraph wrapped over two lines is ONE unit; a table row is its own unit
test_eq(units('one line\nsecond line.\n\n| a | b |\n'), ['one line second line.', '| a | b |'])
test_eq(len(sentences('First. Second.\n\n- a bullet.')), 3)
test_eq(table_rows('| A | B |\n|---|---|\n| 1 | 2 |'), [['A', 'B'], ['1', '2']])
# html outputs are read as text, ANSI colour is stripped
test_eq(output_text(code_cell('x', result(html='<table><tr><td>91.2</td></tr></table>'))).split(),
        ['91.2'])
test_eq(output_text(code_cell('x', stream('\x1b[34m\x1b[1mONNX:\x1b[0m ok'))), 'ONNX: ok')

In [ ]:
#| hide
# Two tokenizers, on purpose. The PROSE one is strict: it decides what a page CLAIMS, so
# it must not invent claims out of file names, versions or identifiers. The HAYSTACK one
# is permissive: it decides what a page SHOWS, and missing a printed number would turn a
# backed claim into a false alarm.
_UNIT = r'%|×|MB|GB|KB|FPS|ms|µs|us|[Mm]illions?|[Bb]illions?|[Tt]housands?|[MKB]\b|x\b|s\b'
_NUM = r'[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?'
NUM_RE = re.compile(r'(?<![\w.\-/])(' + _NUM + r')\s?(' + _UNIT + r')?')
HAY_PLAIN = re.compile(r'\d+(?:\.\d+)?')          # every maximal digit run
HAY_GROUPED = re.compile(r'\d{1,3}(?:,\d{3})+')   # and its thousands-separated reading
_VERSION = re.compile(r'\d+\.\d+(?:\.\d+)+')
_ORDINAL = re.compile(r'(#|§|\bPR\s*|\bv)$')
_WILSON = re.compile(r'Wilson[^.]{0,12}$', re.I)
_SCALES = {'m': 1e6, 'million': 1e6, 'millions': 1e6, 'k': 1e3, 'thousand': 1e3,
           'thousands': 1e3, 'b': 1e9, 'billion': 1e9, 'billions': 1e9}
LATENCY_UNITS = frozenset({'ms', 'µs', 'us', 'FPS'})

def sig_digits(num: str) -> int:
    "Digits of a token once sign, commas, the decimal point and leading zeros are gone"
    return len(num.replace(',', '').replace('.', '').lstrip('-+').lstrip('0'))

def decimals(num: str) -> int: return len(num.partition('.')[2])

def is_checked(num: str, unit: str) -> bool:
    "A prose number is a claim when it carries a unit, a decimal point, or three digits"
    return bool(unit) or '.' in num or sig_digits(num) >= 3

def number_tokens(text: str) -> list[tuple[str, str, int]]:
    "(number, unit, offset) for every claim-shaped number of a prose string"
    skip = [m.span() for m in _VERSION.finditer(text)]
    out = []
    for m in NUM_RE.finditer(text):
        num, unit, i = m.group(1), m.group(2) or '', m.start(1)
        if not is_checked(num, unit): continue
        if any(a <= i < b for a, b in skip): continue                      # 2.9.1, 1.17.0
        if not unit and '.' not in num and re.fullmatch(r'(199\d|20[0-3]\d)', num): continue
        before = text[max(0, i - 14):i]
        if _ORDINAL.search(before): continue                               # #4, §2, PR 51, v3
        if num == '95' and unit == '%' and _WILSON.search(before): continue  # `Wilson 95%`
        out.append((num, unit, i))
    return out

def haystack(nb: dict) -> tuple[set[str], list[float]]:
    "Every number a page shows or runs, as literal strings and as floats"
    text = page_output_text(nb) + '\n' + page_source_text(nb)
    raw = HAY_PLAIN.findall(text) + HAY_GROUPED.findall(text)
    return {r.replace(',', '') for r in raw}, [float(r.replace(',', '')) for r in raw]

def backed(num: str, unit: str, hay: tuple[set[str], list[float]]) -> bool:
    "Is a prose number equal to, or a rounding of, a number the page itself shows?"
    strings, values = hay
    plain = num.replace(',', '').lstrip('+')
    if plain in strings or plain.lstrip('-') in strings: return True
    v, tol = float(plain), max(0.5 * 10 ** -decimals(num), 1e-9)
    scale = _SCALES.get(unit.lower())
    for h in values:
        cands = [h]
        if unit == '%': cands += [h * 100, h / 100]   # fastai prints 0.926928, prose says 92.69%
        if scale: cands.append(h / scale)             # `14.3M` against 14,257,926
        if any(abs(c - v) <= tol for c in cands): return True
    return False

def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    "Wilson score interval for k successes out of n, as proportions"
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z / d * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return max(0.0, c - h), min(1.0, c + h)

In [ ]:
#| hide
# what counts as a claim
test_eq(is_checked('46.7', 'MB'), True)     # unit
test_eq(is_checked('0.89', ''), True)       # decimal: proportions are the charter's currency
test_eq(is_checked('134', ''), True)        # three digits
test_eq(is_checked('32', ''), False)        # `32/32 inputs`, `opset 17`, `batch 1`
test_eq(sig_digits('0.89'), 2)
test_eq(sig_digits('11,689,512'), 8)

_toks = lambda t: [(n, u) for n, u, _ in number_tokens(t)]
test_eq(_toks('torch 2.9.1 and opset 17'), [])                       # version, two digits
test_eq(_toks('published in 2019'), [])                              # year
test_eq(_toks('see #147 and PR 512'), [])                            # ordinals
test_eq(_toks('VGG16 on coco128 with ResNet-18'), [])                # glued to a name
test_eq(_toks('Wilson 95% interval'), [])                            # a method name
test_eq(_toks('baseline **46.7 MB**'), [('46.7', 'MB')])
test_eq(_toks('134 millions parameters and 537MB'), [('134', 'millions'), ('537', 'MB')])
test_eq(_toks('**14.3M** fewer'), [('14.3', 'M')])
test_eq(_toks('a 3.9x smaller file'), [('3.9', 'x')])
test_eq(_toks('counted at 224x224'), [('224', '')])                  # the second 224 is glued
test_eq(_toks('1,478 images'), [('1,478', '')])

# the haystack must survive `torch.randn(1,3,224,224)`, which the strict regex reads as one number
_hay = haystack(mk_nb(code_cell('x = torch.randn(1,3,224,224)'), code_cell('y', stream('11,689,512\n'))))
test_eq('224' in _hay[0], True)
test_eq('11689512' in _hay[0], True)
test_eq(backed('224', '', _hay), True)
test_eq(backed('11,689,512', '', _hay), True)

# a number that only lives in a HIDDEN cell's source is not shown; the same number in a
# visible cell's source is
_hidden_hay = haystack(mk_nb(code_cell('#| hide\nimgsz = 640')))
test_eq('640' in _hidden_hay[0], False)
_visible_hay = haystack(mk_nb(code_cell('model(imgsz=640)')))
test_eq('640' in _visible_hay[0], True)

_h = haystack(mk_nb(code_cell('learn.validate()', stream('0.926928\n46.7 MB\n134309962 params\n'))))
test_eq(backed('92.69', '%', _h), True)     # printed as a proportion, claimed as a percentage
test_eq(backed('46.7', 'MB', _h), True)
test_eq(backed('134', 'million', _h), True) # printed in full, claimed with a scale suffix
test_eq(backed('20', 'x', _h), False)       # a derived ratio nothing printed
test_eq(backed('92.60', '%', _h), False)    # outside the rounding window

test_close(wilson(32, 32), (0.892817, 1.0), eps=1e-5)
test_close([100 * x for x in wilson(1370, 1478)], [91.2524, 93.9119], eps=1e-3)

In [ ]:
#| hide
_TRACEBACK = 'Traceback (most recent call last)'

def check_r1(rel: str, nb: dict) -> list[Violation]:
    "R1 - a committed page shows no failure: no error output, no traceback in a stream"
    out = []
    for i, c in code_cells(nb):
        for o in c.get('outputs') or []:
            if o.get('output_type') == 'error':
                out.append(Violation(rel, 'R1', str(i), f"{o.get('ename', 'error')}: {o.get('evalue', '')}"[:160]))
        if _TRACEBACK in output_text(c):
            out.append(Violation(rel, 'R1', str(i), 'traceback in the cell output'))
    return out

In [ ]:
#| hide
test_eq(len(check_r1('p', mk_nb(code_cell('boom()', error())))), 2)   # error output AND traceback
_handled = 'A dependency reported: it could not convert the graph.\n' + _TRACEBACK + ':\n  ...\n'
test_eq(len(check_r1('p', mk_nb(code_cell('export()', stream(_handled, 'stderr'))))), 1)
# a caught exception printed as a message is not a failure
test_eq(check_r1('p', mk_nb(code_cell('try:\n    f()\nexcept Exception as e:\n    print(e)',
                                      stream('backend x86 cannot quantize INT4\n')))), [])
test_eq(check_r1('p', mk_nb(code_cell('x = 1'))), [])

In [ ]:
#| hide
def check_r10(rel: str, nb: dict) -> list[Violation]:
    "R10 - a committed page carries no stderr stream: library and framework noise is not committed"
    out = []
    for i, c in code_cells(nb):
        for o in c.get('outputs') or []:
            if o.get('output_type') == 'stream' and o.get('name') == 'stderr':
                t = o.get('text') or ''
                text = t if isinstance(t, str) else ''.join(t)
                out.append(Violation(rel, 'R10', str(i), text.strip()[:160]))
    return out

In [ ]:
#| hide
_r10 = lambda src, text: [v.detail for v in check_r10('p', mk_nb(code_cell(src, stream(text, 'stderr'))))]
test_eq(_r10('warn()', 'UserWarning: torchao is experimental\n'), ['UserWarning: torchao is experimental'])
test_eq(_r10('#| hide\nwarn()', 'noise\n'), ['noise'])              # hidden cells are not exempt
test_eq(check_r10('p', mk_nb(code_cell('print(1)', stream('1\n')))), [])   # stdout is fine
test_eq(check_r10('p', mk_nb(code_cell('x = 1'))), [])

In [ ]:
#| hide
_PLACEHOLDER = re.compile(r'^<IPython\.core\.display\.\w+ object>$')
_MAGIC = re.compile(r'^\s*(%%?\w+.*|!.*|#\|.*)$', re.M)

def _substantive(o: dict) -> bool:
    "Does this output put anything on the page?"
    if o.get('output_type') == 'stream': return True
    data = o.get('data') or {}
    if any(k != 'text/plain' for k in data): return True
    tp = data.get('text/plain')
    tp = ''.join(tp) if isinstance(tp, list) else (tp or '')
    return bool(tp.strip()) and not _PLACEHOLDER.match(tp.strip())

def _top_level_print(node: ast.stmt) -> bool:
    "A `print(...)` STATEMENT - a print inside a `def` shows nothing until the def is called"
    return (isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
            and isinstance(node.value.func, ast.Name) and node.value.func.id == 'print')

def _expects_output(source: str) -> bool:
    "Would running this cell put something on the page?"
    body = source.strip()
    if re.match(r'^\s*%%time', body): return True
    try: tree = ast.parse(_MAGIC.sub('', body))
    except SyntaxError: return False
    if not tree.body: return False
    if any(_top_level_print(n) for n in tree.body): return True
    last = tree.body[-1]
    # a trailing bare value is displayed; a trailing CALL may well return None
    return isinstance(last, ast.Expr) and not isinstance(last.value, (ast.Constant, ast.Call))

def check_r2(rel: str, nb: dict) -> list[Violation]:
    "R2 - a visible cell that produces something shows it, and no output is a bare placeholder"
    out = []
    for i, c in visible_code_cells(nb):
        outs = c.get('outputs') or []
        if not outs:
            if _expects_output(src(c)):
                out.append(Violation(rel, 'R2', str(i), 'cell produces a result but carries no output'))
        elif not any(_substantive(o) for o in outs):
            out.append(Violation(rel, 'R2', str(i), 'only placeholder output: ' + output_text(c).strip()[:80]))
    return out

In [ ]:
#| hide
_placeholder = result(plain='<IPython.core.display.HTML object>')
test_eq(len(check_r2('p', mk_nb(code_cell('learn.fit(1)', _placeholder)))), 1)
# the same cell with fastai's metric table beside the placeholder is fine
test_eq(check_r2('p', mk_nb(code_cell('learn.fit(1)', _placeholder,
                                      result(html='<table><tr><td>0.92</td></tr></table>')))), [])
test_eq(len(check_r2('p', mk_nb(code_cell('print(x)')))), 1)          # print, no output
test_eq(check_r2('p', mk_nb(code_cell('x = 1'))), [])                 # assignment only
test_eq(check_r2('p', mk_nb(code_cell('sparsifier.sparsify_model(70)'))), [])  # a call may return None
test_eq(len(check_r2('p', mk_nb(code_cell('model.conv1.weight.ndim')))), 1)    # a bare value IS shown
# a `print` inside a def body prints nothing until the def is called
test_eq(check_r2('p', mk_nb(code_cell('def report(x):\n    print(x)\n'))), [])
test_eq(len(check_r2('p', mk_nb(code_cell('def report(x):\n    return x\n\nprint(report(1))')))), 1)
test_eq(len(check_r2('p', mk_nb(code_cell('%%timeit\nmodel(x)')))), 1)
test_eq(check_r2('p', mk_nb(code_cell('#| hide\nprint(x)'))), [])      # hidden cells are not shown
test_eq(check_r2('p', mk_nb(code_cell('!pip install onnx\nimport onnx'))), [])

In [ ]:
#| hide
# A sentinel is a measurement that failed and printed anyway. It only counts when it is
# pinned to a unit or to a metric label: a bare `0` is a legitimate count, and a library
# log line that rounds a sub-millisecond stage to `0.0ms` is not a failed measurement.
_METRIC = r'(?:model\s+)?(?:size|parameters|params|accuracy|acc|mAP|throughput|memory|latency)'
_SENTINELS = [
    re.compile(r'(?<![\w.])0(?:\.0+)?\s*(?:MB|GB|KB)\b', re.I),
    re.compile(r'(?<![\w.])0\s+parameters\b', re.I),
    re.compile(r'(?<![\w.])(?:nan|inf|-inf)\s*(?:%|MB|GB|KB|ms|µs|us|s|FPS)\b', re.I),
    re.compile(_METRIC + r'[^\n:=]{0,24}[:=]\s*(?:-1|nan|-?inf)(?![\w.])', re.I),
    re.compile(_METRIC + r'[^\n:=]{0,24}[:=]\s*0(?:\.0+)?\s*(?:%|MB|GB|KB)(?![\w.])', re.I),
]

def check_r3(rel: str, nb: dict) -> list[Violation]:
    "R3 - a sentinel (0.00 MB, 0 parameters, nan, -1) is never shown as if it were a result"
    out = []
    for i, c in visible_code_cells(nb):
        text, seen = output_text(c), []
        for rx in _SENTINELS:
            for m in rx.finditer(text):     # a labelled sentinel also matches the bare pattern
                if any(a < m.end() and m.start() < b for a, b in seen): continue
                seen.append(m.span())
                out.append(Violation(rel, 'R3', str(i), m.group(0).strip()))
    return out

In [ ]:
#| hide
_r3 = lambda t: [v.detail for v in check_r3('p', mk_nb(code_cell('x', stream(t))))]
test_eq(_r3('Model size: 0.00 MB\n'), ['0.00 MB'])
test_eq(_r3('Accuracy: -1\n'), ['Accuracy: -1'])
test_eq(_r3('Parameters after pruning: 0 parameters\n'), ['0 parameters'])
test_eq(_r3('mean latency: nan ms\n'), ['nan ms'])
test_eq(_r3("{'n_nonzero_zero_point': 0, 'edges': 8}\n"), [])   # a bare zero is a real count
test_eq(_r3('Speed: 0.0ms preprocess, 61.5ms inference\n'), [])  # rounding in a library log
test_eq(_r3('Model size: 46.7 MB\n'), [])
test_eq(check_r3('p', mk_nb(code_cell('#| hide\nx', stream('size: 0.00 MB')))), [])

In [ ]:
#| hide
# The words a fasterai tutorial does not use. They are all ways of stating a preference or
# a guarantee the page did not measure.
LEXICON = (r'best', r'fastest', r'always', r'never', r'recommend\w*', r'rule of thumb',
           r'key finding', r'tips?', r'guarantee\w*', r'optimal', r'when to use',
           r'for free', r'free speedup', r'no drop in accuracy', r'without any drop')
_LEXICON_RE = re.compile(r'\b(?:' + '|'.join(LEXICON) + r')\b|[✅❌]', re.I)

def check_r5(rel: str, nb: dict, allow: dict | None = None) -> list[Violation]:
    "R5 - the voice: no superlative, no advice word, no verdict emoji, outside code"
    allowed = (LEXICON_ALLOW if allow is None else allow).get(rel, frozenset())
    out = []
    front = frontmatter(nb)
    for key in ('title', 'description'):
        v = front.get(key, '')
        if _LEXICON_RE.search(v) and v not in allowed:
            out.append(Violation(rel, 'R5', 'frontmatter', v))
    for i, md in markdown_cells(nb):
        out += [Violation(rel, 'R5', str(i), s) for s in sentences(prose(md))
                if _LEXICON_RE.search(s) and s not in allowed]
    return out

In [ ]:
#| hide
_bad = 'Tucker is the best general-purpose choice.'
test_eq([v.detail for v in check_r5('p', mk_nb(md_cell(_bad)), {})], [_bad])
test_eq(check_r5('p', mk_nb(md_cell(_bad)), {'p': frozenset({_bad})}), [])  # allow-list is exact
test_eq(check_r5('p', mk_nb(md_cell(_bad)), {'p': frozenset({'Tucker is the best.'})}) != [], True)
# code is not prose: a keyword argument named `always` is not a claim
test_eq(check_r5('p', mk_nb(md_cell("Silence them with `warnings.simplefilter('always')`.")), {}), [])
test_eq(check_r5('p', mk_nb(md_cell("```python\nwarnings.simplefilter('always')\n```")), {}), [])
test_eq(len(check_r5('p', mk_nb(md_cell('| Backend | Best For |\n|---|---|\n| x86 | servers |')), {})), 1)
test_eq(len(check_r5('p', mk_nb(fm(title='T', description='The fastest path')), {})), 1)
test_eq(len(check_r5('p', mk_nb(md_cell('| VGG | ✅ effective |')), {})), 1)
test_eq(check_r5('p', mk_nb(md_cell('This run measures size only.')), {}), [])

In [ ]:
#| hide
R7_REQUIRED = ('title', 'description', 'skip_exec', 'Summary', 'See Also')
_HEADING = {'Summary': re.compile(r'^##\s+Summary\b', re.M),
            'See Also': re.compile(r'^##\s+See Also\b', re.M)}

def check_r7a(rel: str, nb: dict, profile: dict | None = None) -> list[Violation]:
    "R7a - the page carries its front matter and its Summary / See Also sections"
    exempt = (PAGE_PROFILE if profile is None else profile).get(rel, frozenset())
    front, mds = frontmatter(nb), '\n'.join(m for _, m in markdown_cells(nb))
    out = []
    for key in R7_REQUIRED:
        if key in exempt: continue
        if key in _HEADING:
            if not _HEADING[key].search(mds):
                out.append(Violation(rel, 'R7a', 'page', f'no "## {key}" section'))
        elif key == 'skip_exec':
            if front.get('skip_exec', '').lower() != 'true':
                out.append(Violation(rel, 'R7a', 'frontmatter', 'skip_exec is not true'))
        elif not front.get(key):
            out.append(Violation(rel, 'R7a', 'frontmatter', f'no {key}'))
    return out

_LINKS = re.compile(r'!?\[[^\]]*\]\(\s*([^)\s]+)(?:\s+"[^"]*")?\s*\)')
_IMG_EXT = ('.png', '.jpg', '.jpeg', '.gif', '.svg', '.webp')
_DOC_EXT = ('.ipynb', '.qmd', '.md')
RETIRED_SITES = ('nathanhubens.github.io/fasterai',)   # the retired documentation site
_OWN_SITE = 'fasterai-labs.github.io/fasterai/'

def _resolve(nbs: Path, rel: str, target: str) -> Path | None:
    "Where a link points inside `nbs/`, or None when it leaves the site"
    t = target.split('#')[0].split('?')[0].strip()
    if not t or t.startswith(('mailto:', 'tel:')): return None
    low = t.lower()
    if '://' in low:
        if _OWN_SITE in low: t = t[low.index(_OWN_SITE) + len(_OWN_SITE):]
        else: return None
    if not t: return None
    base = nbs if t.startswith('/') else (nbs/rel).parent
    return Path(os.path.normpath(base/t.lstrip('/')))

def _page_with_stem(p: Path) -> bool:
    "Links point at a page STEM, not at a file name; `.html` may come from `.ipynb|.qmd|.md`"
    stem = p.stem.lower()
    return p.parent.is_dir() and any(q.stem.lower() == stem and q.suffix in _DOC_EXT
                                     for q in p.parent.iterdir())

def check_r7b(rel: str, nb: dict, nbs: Path) -> list[Violation]:
    "R7b - every internal link resolves to a page or an image that exists"
    out = []
    for i, md in markdown_cells(nb):
        for target in _LINKS.findall(md):
            low = target.lower()
            if any(h in low for h in RETIRED_SITES):
                out.append(Violation(rel, 'R7b', str(i), f'links to the retired site: {target}')); continue
            p = _resolve(nbs, rel, target)
            if p is None: continue
            if p.suffix == '.html':
                if not _page_with_stem(p): out.append(Violation(rel, 'R7b', str(i), f'dead page link: {target}'))
            elif p.suffix.lower() in _IMG_EXT and not p.exists():
                out.append(Violation(rel, 'R7b', str(i), f'missing image: {target}'))
            elif p.suffix in _DOC_EXT and not p.exists():
                out.append(Violation(rel, 'R7b', str(i), f'dead page link: {target}'))
    return out

In [ ]:
#| hide
_NBS = nbs_dir()
_ok = mk_nb(fm(title='T', description='D', skip_exec='true'),
            md_cell('## Summary\n\n## See Also\n'))
test_eq(check_r7a('p', _ok, {}), [])
test_eq([v.detail for v in check_r7a('p', mk_nb(fm(title='T', skip_exec='true'),
                                                md_cell('## Summary')), {})],
        ['no description', 'no "## See Also" section'])
# the landing page runs in CI and carries no Summary of its own
test_eq(check_r7a('overview.ipynb', mk_nb(fm(title='Overview'), md_cell('hi')),
                  {'overview.ipynb': frozenset({'description', 'skip_exec', 'Summary', 'See Also'})}), [])

_lk = lambda t, rel='tutorials/quantize/deployable_export.ipynb': \
    [v.detail for v in check_r7b(rel, mk_nb(md_cell(f'see [x]({t})')), _NBS)]
test_eq(_lk('../../core/precision.html'), [])            # resolves to nbs/core/precision.ipynb
test_eq(_lk('quantization_compared.html'), [])           # a sibling page
test_eq(_lk('../prune/yolov8.html'), [])                 # links point at stems, case aside
test_eq(_lk('/index.html'), [])                          # nbs/index.qmd
test_eq(_lk('#a-section'), [])                           # a bare anchor
test_eq(_lk('https://pytorch.org/docs/stable/quantization.html'), [])
test_eq(_lk('mailto:someone@example.com'), [])
test_eq(_lk('tutorial.sparsifier.html'), ['dead page link: tutorial.sparsifier.html'])
test_eq(_lk('https://nathanhubens.github.io/fasterai/granularity.html'),
        ['links to the retired site: https://nathanhubens.github.io/fasterai/granularity.html'])
test_eq(_lk('imgs/does_not_exist.png'), ['missing image: imgs/does_not_exist.png'])
test_eq(_lk('../../core/precision.html#support'), [])    # the anchor is dropped first

In [ ]:
#| hide
_SIDEBAR = re.compile(r'^\s*-\s*(\S+\.ipynb)\s*$', re.M)
CONTRACT_PAGE = 'tutorial_contract.ipynb'

def check_r8(sidebar_yml: str, pages: list[str], on_disk: set[str],
             unlisted_ok=frozenset()) -> list[Violation]:
    "R8 - the sidebar and the tutorials folder agree, in both directions"
    listed = set(_SIDEBAR.findall(sidebar_yml))
    out = [Violation(p, 'R8', 'sidebar', 'page is not in _quarto.yml')
           for p in sorted(pages) if p not in listed and p not in unlisted_ok]
    out += [Violation(e, 'R8', 'sidebar', 'sidebar entry does not exist on disk')
            for e in sorted(listed) if e.startswith('tutorials/') and e not in on_disk]
    if CONTRACT_PAGE not in listed:
        out.append(Violation(CONTRACT_PAGE, 'R8', 'sidebar', 'the contract page is not in _quarto.yml'))
    return out

In [ ]:
#| hide
_yml = ('    contents:\n      - tutorials/a.ipynb\n      - tutorials/gone.ipynb\n'
        '      - tutorial_contract.ipynb\n')
test_eq([(v.page, v.detail) for v in check_r8(_yml, ['tutorials/a.ipynb', 'tutorials/b.ipynb'],
                                              {'tutorials/a.ipynb', 'tutorials/b.ipynb'})],
        [('tutorials/b.ipynb', 'page is not in _quarto.yml'),
         ('tutorials/gone.ipynb', 'sidebar entry does not exist on disk')])
test_eq(check_r8('      - tutorials/a.ipynb\n      - tutorial_contract.ipynb\n',
                 ['tutorials/a.ipynb'], {'tutorials/a.ipynb'}), [])
# the contract page must be reachable from the sidebar too
test_eq([v.detail for v in check_r8('      - tutorials/a.ipynb\n', ['tutorials/a.ipynb'],
                                    {'tutorials/a.ipynb'})],
        ['the contract page is not in _quarto.yml'])
test_eq(check_r8('      - tutorial_contract.ipynb\n', ['tutorials/b.ipynb'], {'tutorials/b.ipynb'},
                 unlisted_ok=frozenset({'tutorials/b.ipynb'})), [])

In [ ]:
#| hide
# R9 has two halves, both scoped to a SENTENCE so a number two clauses away never gets
# swept in. The REQUIREMENT: a sentence that states an accuracy carries the n it was
# measured on. The VERIFIER: a k/n is recomputed against the percentage printed right next
# to it - never one merely sharing the sentence - and against an adjacent interval; the
# numbers it proves are handed to R4, which would otherwise ask the page to print its own
# confidence bounds.
_ACC_WORD = re.compile(r'\b(?:accuracy|accuracies|accurate|agreement|agrees?|top-1|error rate)\b', re.I)
_INT = r'(?:\d{1,3}(?:,\d{3})+|\d+)'
_HAS_N = re.compile(r'\bn\s*=\s*' + _INT
                    + r'|\b' + _INT + r'\s+(?:validation|test|held-out|evaluation|probe)\b')
_PCT = re.compile(r'(?<![\w.\-/])(' + _INT + r'(?:\.\d+)?)\s?%')
_FRAC = re.compile(r'(?<![\w.\-/])(' + _INT + r')\s*(?:/|\s+(?:correct\s+)?out of\s+)\s*('
                   + _INT + r')(?![\d./])')
_BRACKET = re.compile(r'\[\s*(\d+(?:\.\d+)?)\s*,\s*(\d+(?:\.\d+)?)\s*\]')
_PROPORTION = re.compile(r'(?<![\w.\-/])0\.\d{2,}')

# A k/n pairs only with the percentage printed immediately next to it - never one merely
# sharing the sentence. `_CONNECT` absorbs the markdown noise (`**`, spaces) in between.
_CONNECT = r'[\s*_]{0,4}'
_PCT_TOK = r'(' + _INT + r'(?:\.\d+)?)\s?%'
_FRAC_TOK = r'(' + _INT + r')\s*/\s*(' + _INT + r')'
_PAIR_BEFORE = re.compile(_PCT_TOK + _CONNECT + r'\(' + _CONNECT + _FRAC_TOK)
_PAIR_AFTER = re.compile(_FRAC_TOK + _CONNECT + r'[=(]' + _CONNECT + _PCT_TOK)

def _paired_pct(s: str) -> dict[tuple[str, str], str]:
    "k/n -> the percentage printed right before or right after it, if any"
    out = {}
    for m in _PAIR_BEFORE.finditer(s): out[(m.group(2), m.group(3))] = m.group(1)
    for m in _PAIR_AFTER.finditer(s): out[(m.group(1), m.group(2))] = m.group(3)
    return out

def _percents(s: str) -> list[str]:
    "The percentages a unit claims - the 95 of `Wilson 95%` is a method name, not a claim"
    return [m.group(1) for m in _PCT.finditer(s)
            if not (m.group(1) == '95' and _WILSON.search(s[max(0, m.start(1) - 14):m.start(1)]))]

def check_r9(rel: str, nb: dict) -> tuple[list[Violation], dict[int, set[str]]]:
    "R9 - accuracy sentences carry their n; every k/n and every interval in prose is recomputed"
    out: list[Violation] = []
    verified: dict[int, set[str]] = {}
    for i, md in markdown_cells(nb):
        ok = verified.setdefault(i, set())
        for s in sentences(prose(md)):
            pcts = _percents(s)
            fracs = [(m.group(1), m.group(2)) for m in _FRAC.finditer(s)]
            brs = [(m.group(1), m.group(2)) for m in _BRACKET.finditer(s)]
            paired = _paired_pct(s)
            if _ACC_WORD.search(s) and (pcts or _PROPORTION.search(s)) \
                    and not (_HAS_N.search(s) or fracs):
                out.append(Violation(rel, 'R9', str(i), s))
            for j, (ks, ns) in enumerate(fracs):
                k, n = int(ks.replace(',', '')), int(ns.replace(',', ''))
                if n == 0 or k > n: continue
                good = True
                ps = paired.get((ks, ns))
                if ps is not None:
                    p, tol = float(ps.replace(',', '')), max(0.5 * 10 ** -decimals(ps), 1e-9)
                    if abs(100.0 * k / n - p) > tol:
                        out.append(Violation(rel, 'R9', str(i),
                                             f'{ps}% is not {ks}/{ns} = {100.0 * k / n:.4f}%'))
                        good = False
                    else: ok.add(ps)
                if j < len(brs):
                    a_s, b_s = brs[j]
                    a, b = float(a_s), float(b_s)
                    scale = 100.0 if max(a, b) > 1.0 else 1.0      # percent or proportion
                    lo, hi = (x * scale for x in wilson(k, n))
                    if abs(lo - a) > 0.01 or abs(hi - b) > 0.01:   # 0.01 on the printed scale
                        out.append(Violation(rel, 'R9', str(i), f'[{a_s}, {b_s}] is not the Wilson '
                                             f'interval of {ks}/{ns} = [{lo:.4f}, {hi:.4f}]'))
                        good = False
                    else: ok.update((a_s, b_s))
                if good: ok.update((ks, ns))
    return out, verified

def r9_verified(nb: dict) -> dict[int, set[str]]:
    "Cell index -> the numeric tokens R9 proved, so R4 does not ask the page to print them"
    return check_r9('', nb)[1]

In [ ]:
#| hide
# The four sentence forms that already exist in these tutorials. They must all pass, and
# their numbers must come back verified - the earlier draft of this rule failed all four.
REAL_SENTENCES = {
 'regularize': 'In this run the regularized model ends at **92.69%** (1370/1478) against '
               '**92.02%** (1360/1478) for the baseline — one run each, with Wilson 95% '
               'intervals [91.25, 93.91] and [90.52, 93.29] that overlap.',
 'walkthrough': 'With the teacher, the student ends at **81.38%** (3194/3925, Wilson 95% '
                '[80.13, 82.56]) against **81.17%** (3186/3925, [79.92, 82.36]) for the '
                'vanilla **VGG16** — one run each, and the two intervals overlap.',
 'export': '32/32 inputs agree — as a proportion, 1.000, with a Wilson 95% interval of\n'
           '[0.89, 1.00]. Read it for exactly what it is:',
 'folding': '| Accuracy (n = 1478, one run) | 79.2963% | 79.2963% | 1172/1478 in both cases |',
}
for name, s in REAL_SENTENCES.items():
    vs, ok = check_r9('p', mk_nb(md_cell(s)))
    test_eq((name, vs), (name, []))
    test_eq((name, len(ok[0]) > 0), (name, True))
test_eq(check_r9('p', mk_nb(md_cell(REAL_SENTENCES['walkthrough'])))[1][0],
        {'81.38', '3194', '3925', '80.13', '82.56', '81.17', '3186', '79.92', '82.36'})

# the requirement: an accuracy with no n
test_eq(len(check_r9('p', mk_nb(md_cell('The pruned model reaches an accuracy of 92.7%.')))[0]), 1)
test_eq(check_r9('p', mk_nb(md_cell('Accuracy 92.69% (1370/1478).')))[0], [])
test_eq(check_r9('p', mk_nb(md_cell('Both accuracies are measured on the 1478 validation images '
                                    'and printed with their Wilson 95% interval.')))[0], [])
test_eq(check_r9('p', mk_nb(md_cell('The mAP drops to 0.4797 on this run.')))[0], [])  # not binomial

# the verifier: a percentage that is not k/n, and an interval that is not Wilson
test_eq([v.detail for v in check_r9('p', mk_nb(md_cell('accuracy 92.60% (1370/1478)')))[0]],
        ['92.60% is not 1370/1478 = 92.6928%'])
test_eq([v.detail for v in check_r9('p', mk_nb(md_cell('accuracy 92.69% (1370/1478), '
                                                       'Wilson 95% [80.00, 90.00]')))[0]],
        ['[80.00, 90.00] is not the Wilson interval of 1370/1478 = [91.2524, 93.9119]'])

# the printed form is `k/n = xx.xx% [lo, hi]`; a wrong percentage right after a k/n
# still fails, whichever side of the fraction it prints on
test_eq(check_r9('p', mk_nb(md_cell('sparsified 1374/1478 = 92.96% [91.55, 94.16].')))[0], [])
test_eq([v.detail for v in check_r9('p', mk_nb(md_cell('sparsified 1374/1478 = 50.00%.')))[0]],
        ['50.00% is not 1374/1478 = 92.9635%'])

# a percentage merely sharing the sentence - not printed next to the k/n - does not pair
# with it: the false positive this fix closes (sensitivity.ipynb, sparsify_callback.ipynb)
test_eq(check_r9('p', mk_nb(md_cell('The target is 50.00% overall, and the model scores '
                                    '1374/1478 after.')))[0], [])

# an accuracy word and an unrelated percentage in the SAME PARAGRAPH but a DIFFERENT
# sentence do not trigger the requirement either (walkthrough.ipynb's false positive)
test_eq(check_r9('p', mk_nb(md_cell('The model ends the recovery fit at 50 % zeros again. '
                                    'The accuracy after recovery sits inside the '
                                    "baseline's interval.")))[0], [])

In [ ]:
#| hide
SCOPE_PHRASES = ('single run', 'one run', 'this run', 'measured on')

def check_r4(rel: str, nb: dict) -> list[Violation]:
    "R4 - every number the prose claims is a number the page itself prints"
    hay, verified, out = haystack(nb), r9_verified(nb), []
    for i, md in markdown_cells(nb):
        ok = verified.get(i, set())
        for num, unit, _ in number_tokens(prose(md)):
            if num in ok: continue                      # R9 proved this one arithmetically
            if not backed(num, unit, hay):
                out.append(Violation(rel, 'R4', str(i), (num + unit).strip()))
    return out

def check_r4b(rel: str, nb: dict) -> list[Violation]:
    "R4b - a page that reports a latency says which run it came from"
    text = '\n'.join(prose(md) for _, md in markdown_cells(nb))
    if not any(u in LATENCY_UNITS for _, u, _ in number_tokens(text)): return []
    if any(ph in text.lower() for ph in SCOPE_PHRASES): return []
    return [Violation(rel, 'R4b', 'page', 'reports a latency with no scope line (one of: '
                      + ', '.join(SCOPE_PHRASES) + ')')]

In [ ]:
#| hide
_out = code_cell('bench()', stream('46.7 MB\n0.926928\n37,194,430 -> 22,936,504\n'))
_r4 = lambda text: [v.detail for v in check_r4('p', mk_nb(md_cell(text), _out))]
test_eq(_r4('The baseline is **46.7 MB**.'), [])
test_eq(_r4('It validates at **92.69%**.'), [])            # printed as a proportion
test_eq(_r4('Exported with torch 2.9.1 at opset 17.'), []) # versions are not claims
test_eq(_r4('It is **20x** smaller.'), ['20x'])            # a ratio nothing printed
test_eq(_r4('**14.3M** fewer parameters.'), ['14.3M'])     # a delta nothing printed
# R9's verified numbers are exempt: a page must not print its own confidence bounds
test_eq(check_r4('p', mk_nb(md_cell('32/32 inputs agree — as a proportion, 1.000, with a Wilson '
                                    '95% interval of [0.89, 1.00].'),
                            code_cell('verify()', stream('1.000\n32\n')))), [])

# a VISIBLE code cell's source is "shown": an argument backs the same claim in prose
test_eq(check_r4('p', mk_nb(md_cell('It runs at `imgsz` **640**.'),
                            code_cell("model.val(imgsz=640)"))), [])
# the same number in a HIDDEN cell's source is not shown, and must still be flagged
test_eq([v.detail for v in check_r4('p', mk_nb(md_cell('It runs at `imgsz` **640**.'),
                                                code_cell("#| hide\nmodel.val(imgsz=640)")))],
        ['640'])

_lat = lambda text: [v.rule for v in check_r4b('p', mk_nb(md_cell(text)))]
test_eq(_lat('The folded model runs in **1.19 ms**.'), ['R4b'])
test_eq(_lat('The folded model runs in **1.19 ms** (single run, CPU).'), [])
test_eq(_lat('The model is **46.7 MB**.'), [])             # size is not a latency

In [ ]:
#| hide
# The vocabularies come from the library itself, never from a hand-kept list of strings:
# a list would drift, and a name that only lives in a list is exactly the defect R6 hunts.
import importlib, inspect, pkgutil
import fasterai
from fasterai.core.criteria import Criteria
from fasterai.core.schedule import Schedule
from fasterai.core.granularity import Granularities
from fasterai.core.precision import _BACKENDS, _LEGACY_BACKENDS

def public_api() -> dict[str, object]:
    "name -> object for every name fasterai exports, by importing the whole package"
    names: dict[str, object] = {}
    for mi in pkgutil.walk_packages(fasterai.__path__, prefix='fasterai.'):
        if mi.name.rsplit('.', 1)[-1] in ('all', '_modidx'): continue
        mod = importlib.import_module(mi.name)   # a failure here fails the harness, on purpose
        for a in getattr(mod, '__all__', ()): names.setdefault(a, getattr(mod, a, None))
    return names

NAME2OBJ = public_api()
PUBLIC = frozenset(NAME2OBJ)
CRITERIA = frozenset(n for n, o in NAME2OBJ.items() if isinstance(o, Criteria))
SCHEDULES = frozenset(n for n, o in NAME2OBJ.items() if isinstance(o, Schedule))
GRANULARITIES = frozenset(k for d in Granularities._granularities.values() for k in d)
LOSSES = frozenset(importlib.import_module('fasterai.distill.losses').__all__)
BACKENDS = frozenset(_BACKENDS) | frozenset(_LEGACY_BACKENDS)
VOCAB = (('granularit', GRANULARITIES), ('criteri', CRITERIA), ('schedul', SCHEDULES),
         ('backend', BACKENDS), ('loss', LOSSES))
_NOUN = {'granularit': 'granularity', 'criteri': 'criteria', 'schedul': 'schedule',
         'backend': 'backend', 'loss': 'distillation loss'}

test_eq({'large_final', 'movement', 'wanda'} <= CRITERIA, True)
test_eq('movmag' in CRITERIA, False)          # it is in `criterias` but it is not importable
test_eq({'one_shot', 'agp', 'iterative'} <= SCHEDULES, True)
test_eq({'weight', 'kernel', 'filter'} <= GRANULARITIES, True)
test_eq('vector' in GRANULARITIES, False)
test_eq({'SoftTarget', 'Attention'} <= LOSSES, True)
test_eq({'x86', 'pt2e', 'torchao'} <= BACKENDS, True)

In [ ]:
#| hide
_DROP = re.compile(r'^\s*(\.\.\.|[%!].*)\s*$', re.M)
_ELLIPSIS = ((r'(?<![.\w])\.\.\.(?![.\w])', ''), (r',\s*,', ','), (r'\(\s*,', '('), (r',\s*\)', ')'))
_IDENT = re.compile(r'^[A-Za-z_]\w*$')
_ROW_KEY = re.compile(r'(?:\w+\s+)?(granularity|criteria|schedule|backend|loss)(?:\s+\w+)?', re.I)

def _parseable(block: str) -> str:
    "Doc blocks elide arguments with `...`; drop those and the magics before parsing"
    b = _DROP.sub('', block)
    for pat, rep in _ELLIPSIS: b = re.sub(pat, rep, b)
    return b

def _norm_span(s: str) -> str:
    s = re.sub(r'\(.*\)$', '', s.strip().strip('*').strip()).strip()
    return s.strip('\'"')

def _row_key(cell: str):
    "The vocabulary a table row is about, when its first cell is that word and nothing else"
    m = _ROW_KEY.fullmatch(cell.strip().strip('*`').strip())
    if not m: return None
    word = m.group(1).lower()
    return next(((k, v) for k, v in VOCAB if word.startswith(k)), None)

def _callee(name2obj: dict, node: ast.Call):
    "(label, object) for a call whose callee is a fasterai name, else (None, None)"
    f = node.func
    if isinstance(f, ast.Name):
        if f.id == 'partial' and node.args and isinstance(node.args[0], ast.Name) \
                and node.args[0].id in name2obj:
            return f'partial({node.args[0].id})', name2obj[node.args[0].id]
        if f.id in name2obj: return f.id, name2obj[f.id]
    if isinstance(f, ast.Attribute):
        v = f.value
        owner_name = (v.func.id if isinstance(v, ast.Call) and isinstance(v.func, ast.Name)
                      else v.id if isinstance(v, ast.Name) else None)
        owner = name2obj.get(owner_name)
        if inspect.isclass(owner) and hasattr(owner, f.attr):
            return f'{owner_name}.{f.attr}', getattr(owner, f.attr)
    return None, None

def _check_calls(rel: str, i: int, block: str, name2obj: dict) -> list[Violation]:
    try: tree = ast.parse(_parseable(block))
    except SyntaxError as e: return [Violation(rel, 'R6', str(i), f'python block does not parse: {e.msg}')]
    out = []
    for node in ast.walk(tree):
        if not isinstance(node, ast.Call): continue
        label, obj = _callee(name2obj, node)
        if obj is None: continue
        if not callable(obj):
            out.append(Violation(rel, 'R6', str(i), f'{label} is not callable')); continue
        try: sig = inspect.signature(obj)
        except (ValueError, TypeError): continue
        if any(p.kind is inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values()): continue
        out += [Violation(rel, 'R6', str(i), f'{label}({kw.arg}=...): {label} has no such parameter')
                for kw in node.keywords if kw.arg and kw.arg not in sig.parameters]
    return out

def _check_tables(rel: str, i: int, md: str) -> list[Violation]:
    "A table that names a vocabulary must spell its members the way the library does"
    out, rows = [], table_rows(md)
    if not rows: return out
    header = [c.lower() for c in rows[0]]
    cols = {j: kv for j, h in enumerate(header) for kv in VOCAB if kv[0] in h}
    for row in rows[1:]:
        checks = [(row[j], *cols[j]) for j in range(len(row)) if j in cols]
        kv = _row_key(row[0]) if row else None
        if kv: checks += [(c, *kv) for c in row[1:]]
        for cell, key, voc in checks:
            out += [Violation(rel, 'R6', str(i), f'`{n}` is not a fasterai {_NOUN[key]}')
                    for n in map(_norm_span, code_spans(cell))
                    if _IDENT.match(n) and n not in voc and n not in PUBLIC]
    return out

def check_r6(rel: str, nb: dict, phantoms=None, name2obj: dict | None = None) -> list[Violation]:
    "R6 - names shown in prose exist: blocks parse, keywords are real, vocabularies are real"
    name2obj = NAME2OBJ if name2obj is None else name2obj
    deny = [(p, re.compile(p)) for p in (KNOWN_PHANTOMS if phantoms is None else phantoms)]
    out = []
    for i, md in markdown_cells(nb):
        blocks = python_blocks(md)
        for b in blocks: out += _check_calls(rel, i, b, name2obj)
        out += _check_tables(rel, i, md)
        for text in [*blocks, *code_spans(md)]:
            out += [Violation(rel, 'R6', str(i), f'{m.group(0)!r} (denylist /{pat}/)')
                    for pat, rx in deny for m in rx.finditer(text)]
    return out

In [ ]:
#| hide
_blk = lambda code: mk_nb(md_cell('```python\n' + code + '\n```'))
_r6 = lambda nb, ph=(): [v.detail for v in check_r6('p', nb, phantoms=ph)]

# keyword arguments, checked against the real signature
test_eq(_r6(_blk("SparsifyCallback(50, 'weight', 'local', large_final, iterative, start_epoch=5)")),
        ['SparsifyCallback(start_epoch=...): SparsifyCallback has no such parameter'])
test_eq(_r6(_blk("Conv_Decomposer().decompose(model, 0.5, data=[b])")),
        ['Conv_Decomposer.decompose(data=...): Conv_Decomposer.decompose has no such parameter'])
test_eq(_r6(_blk("FC_Decomposer().decompose(model, 0.5, data=[b])")), [])   # `data` is real there
test_eq(_r6(_blk("Sparsifier(model, 'weight', 'local', large_final, data=dl)")), [])
test_eq(_r6(_blk("partial(iterative, n_steps=5)")), ['partial(iterative) is not callable'])
test_eq(_r6(_blk("SparsifyCallback(sparsity=50, granularity='weight', ...)")), [])  # `...` is elided
test_eq(_r6(_blk("learn.fit_one_cycle(5, cbs=sp_cb)")), [])                 # not a fasterai name
test_eq(_r6(_blk("def f(:\n    pass")), ['python block does not parse: invalid syntax'])

# vocabulary tables
_tbl = '| Parameter | Description | Example |\n|---|---|---|\n'
test_eq(_r6(mk_nb(md_cell(_tbl + '| `criteria` | Importance | `large_final`, `magnitude` |'))),
        ['`magnitude` is not a fasterai criteria'])
test_eq(_r6(mk_nb(md_cell(_tbl + "| `granularity` | Level | `'weight'`, `'vector'` |"))),
        ['`vector` is not a fasterai granularity'])
test_eq(_r6(mk_nb(md_cell(_tbl + "| `schedule` | Ramp | `one_cycle`, `cos`, `lin` |"))), [])
# a row whose first cell merely MENTIONS a vocabulary word is prose, not a vocabulary row
test_eq(_r6(mk_nb(md_cell('| Tool | What it gives you |\n|---|---|\n'
                          '| `KnowledgeDistillationCallback(teacher, loss, schedule)` | added during `fit` |'))), [])

# the denylist: names the structural checks cannot refute (`Pruner` takes **kwargs)
_deny = (r'Pruner\([^)]*\bsparsity\s*=',)
test_eq(_r6(_blk("Pruner(learn.model, sparsity=0.3, context='local')"), _deny),
        ["'Pruner(learn.model, sparsity=' (denylist /Pruner\\([^)]*\\bsparsity\\s*=/)"])
test_eq(_r6(_blk("Pruner(learn.model, 50, 'local', large_final)"), _deny), [])
test_eq(_r6(_blk("Pruner(learn.model, sparsity=0.3)")), [])   # no signature can refute **kwargs

In [ ]:
#| hide
# Two rules cannot be waived because a waiver would hide a broken page: a failure shown as
# a result (R1, R3) and a link or a sidebar entry that goes nowhere (R7b, R8).
NON_QUARANTINABLE = frozenset({'R1', 'R3', 'R7b', 'R8'})

# Names that appeared in these pages and do not exist in the library. The structural checks
# cannot refute all of them - `Pruner` and `PruneCallback` accept `**kwargs`, so no signature
# can prove `sparsity=` wrong - so this is the regression net. Entries are added when a
# reviewer finds a name the checks above cannot refute; they are never removed.
KNOWN_PHANTOMS = (
    r'\bvector\b', r'\bmagnitude\b', r'\btaylor\b', r'\blinear\b', r'\bRKD\b', r'\bPKT\b',
    r'\bstart_epoch\b', r'\bqconfig\b',
    r'Pruner\([^)]*\bsparsity\s*=', r'PruneCallback\([^)]*\bsparsity\s*=',
    r'PruneCallback\(\s*sparsity\b', r'partial\(\s*iterative\b',
    r'Conv_Decomposer\(\)\.decompose\([^)]*\bdata\s*=',
)

# `overview.ipynb` is the landing page: it executes in CI and has no Summary of its own.
PAGE_PROFILE = {'overview.ipynb': frozenset({'description', 'skip_exec', 'Summary', 'See Also'})}

UNLISTED_OK: frozenset[str] = frozenset()          # pages allowed to be absent from the sidebar
LEXICON_ALLOW: dict[str, frozenset[str]] = {}      # page -> sentences allowed under R5, verbatim
N_PAGES = 21                                       # pinned: a page added or lost is a review event

In [ ]:
#| hide
# The worklist: what each page is still allowed to fail, computed once by running the
# checks over the tree this notebook landed on. Rewriting a page means deleting its entry
# in the same commit - the stale guard below fails the build if an entry stops being used.
#
# Recomputed against the tree this branch integrates: docs/tutorials-sparse-rerun,
# docs/tutorials-quantize-export-modernize, docs/tutorials-walkthrough-quickstart and
# docs/tutorials-prune-misc-rerun. Empty: the R9/R4/R10 fixes cleared four pages outright
# without touching their prose, and the last four entries - quickstart.ipynb's missing
# `## See Also`, and the conv_decomposer/YOLOV8/pruner numbers that were absent from every
# cell - were cleared by the amended `docs/tutorials-walkthrough-quickstart` (acbff65) and
# `docs/tutorials-prune-misc-rerun` (ba548da).
QUARANTINE: dict[str, frozenset[str]] = {}

In [ ]:
#| hide
# The frozen copy of QUARANTINE taken when this page landed. It is the ceiling: nothing may
# be added to QUARANTINE that is not already here. Edited by review only.
_LANDED: dict[str, frozenset[str]] = {}

# No violation of an unwaivable rule (R1, R3, R7b, R8) is open on the tree this notebook
# landed on: the two entries this dict used to carry (YOLOV8's ONNX traceback, sparsifier's
# retired-site links) were both cleared by the branches this commit integrates.
OPEN_AT_LANDING = {}
test_eq(set(OPEN_AT_LANDING) & {p for p, r in QUARANTINE.items() if r & NON_QUARANTINABLE}, set())

In [ ]:
#| hide
def check_page(rel: str, nb: dict, nbs: Path) -> list[Violation]:
    "Every per-page rule, in rule order"
    return [*check_r1(rel, nb), *check_r2(rel, nb), *check_r3(rel, nb), *check_r4(rel, nb),
            *check_r4b(rel, nb), *check_r5(rel, nb), *check_r6(rel, nb), *check_r7a(rel, nb),
            *check_r7b(rel, nb, nbs), *check_r9(rel, nb)[0], *check_r10(rel, nb)]

def run_gate(nbs: Path) -> tuple[list[str], list[Violation]]:
    "Discover the pages, run every rule, and check the sidebar once"
    pages = discover_pages(nbs)
    vs = [v for rel in pages for v in check_page(rel, load_page(nbs, rel), nbs)]
    vs += check_r8((nbs/'_quarto.yml').read_text(), pages, set(pages), UNLISTED_OK)
    return pages, vs

def gate_problems(pages: list[str], violations: list[Violation], quarantine: dict,
                  landed: dict, on_disk: set[str], n_pages: int | None = None) -> list[str]:
    "Everything that would make the quarantine itself dishonest"
    n_pages = N_PAGES if n_pages is None else n_pages
    problems = []
    if len(pages) != n_pages:
        problems.append(f'{len(pages)} pages discovered, N_PAGES says {n_pages}')
    for page, rules in quarantine.items():
        if page not in on_disk: problems.append(f'{page}: quarantined page is not on disk')
        if page not in landed: problems.append(f'{page}: new page in QUARANTINE; it may only shrink')
        elif not set(rules) <= set(landed[page]):
            problems.append(f'{page}: {sorted(set(rules) - set(landed[page]))} was not quarantined at landing')
        blocked = sorted(set(rules) & NON_QUARANTINABLE)
        if blocked: problems.append(f'{page}: {blocked} cannot be quarantined')
    seen = {(v.page, v.rule) for v in violations}
    for page, rules in quarantine.items():
        problems += [f'{page} now passes {r}: delete the entry from QUARANTINE'
                     for r in sorted(rules) if (page, r) not in seen]
    return problems

def remaining(violations: list[Violation], quarantine: dict) -> list[Violation]:
    return [v for v in violations if v.rule not in quarantine.get(v.page, ())]

In [ ]:
#| hide
# The quarantine is a worklist that can only shrink. These tests are the gate on the gate.
_pages = ['tutorials/a.ipynb', 'tutorials/b.ipynb']
_disk = set(_pages)
_vs = [Violation('tutorials/a.ipynb', 'R5', '3', 'the best choice'),
       Violation('tutorials/a.ipynb', 'R4', '3', '20x')]
_landed = {'tutorials/a.ipynb': frozenset({'R4', 'R5'})}
_probe = lambda q, pages=_pages: gate_problems(pages, _vs, q, _landed, _disk, len(_pages))

_full = {'tutorials/a.ipynb': frozenset({'R4', 'R5'})}
test_eq(_probe(_full), [])
test_eq(remaining(_vs, _full), [])
# a page that is not in _LANDED cannot be added
test_ne(_probe({**_full, 'tutorials/b.ipynb': frozenset({'R5'})}), [])
# a rule that was not quarantined at landing cannot be added to an existing page
test_ne(_probe({'tutorials/a.ipynb': frozenset({'R4', 'R5', 'R6'})}), [])
# nothing non-quarantinable may enter, even if it is in _LANDED
test_ne(gate_problems(_pages, _vs, {'tutorials/a.ipynb': frozenset({'R1'})},
                      {'tutorials/a.ipynb': frozenset({'R1'})}, _disk, len(_pages)), [])
# a key that no longer exists on disk must be deleted
test_ne(_probe({**_full, 'tutorials/gone.ipynb': frozenset({'R5'})}), [])
# the stale guard: once a page passes a rule, its entry has to go
test_eq(['tutorials/a.ipynb now passes R4: delete the entry from QUARANTINE'],
        gate_problems(_pages, [_vs[0]], _full, _landed, _disk, len(_pages)))
# a page count that moved is a review event
test_ne(_probe(_full, _pages + ['tutorials/c.ipynb']), [])
# quarantining hides only the rule it names, on the page it names
test_eq([str(v.rule) for v in remaining(_vs, {'tutorials/a.ipynb': frozenset({'R5'})})], ['R4'])

In [ ]:
#| hide
#| tutorials
# The execution gate. It runs every tutorial in a throw-away copy of its folder and reports
# what the code no longer reproduces. It is NOT part of the default suite: it needs a GPU,
# the datasets and the optional dependencies, and it takes about an hour.
#
#   python -c "from nbdev.test import nbdev_test; nbdev_test(path='nbs/tutorial_contract.ipynb', flags='tutorials')"
#
# `FASTERAI_TUTORIALS` narrows it to a glob, e.g. FASTERAI_TUTORIALS='tutorials/sparse/*'.
import fnmatch, shutil, sys, tempfile, time
import nbformat
from nbclient import NotebookClient
from jupyter_client.manager import KernelManager

BUDGET = {'tutorials/walkthrough.ipynb': 1800, 'tutorials/prune/YOLOV8.ipynb': 1200,
          'tutorials/sparse/transformers.ipynb': 900}
DEFAULT_BUDGET = 600
DRIFT_FAILS = re.compile(r'(?:MB|GB|KB|x)$')   # sizes and ratios are deterministic

def _kernel_manager() -> KernelManager:
    "A kernel that is THIS interpreter, whatever the installed `python3` spec points at"
    km = KernelManager(kernel_name='python3')
    km.kernel_spec.argv[0] = sys.executable
    return km

def _guard(worktree: Path):
    "Prepended to every run: the kernel must import the fasterai under test"
    return nbformat.v4.new_code_cell(
        'import fasterai, pathlib\n'
        f'assert pathlib.Path(fasterai.__file__).is_relative_to(pathlib.Path({str(worktree)!r})), '
        'fasterai.__file__\n')

def execute_page(nbs: Path, rel: str, run_dir: Path):
    "Run one page with its folder copied out of the repo, so nothing is written under nbs/"
    dst = run_dir/Path(rel).parent
    if not dst.exists(): shutil.copytree(nbs/Path(rel).parent, dst)
    nb = nbformat.read(nbs/rel, as_version=4)
    nb.cells.insert(0, _guard(nbs.parent))
    NotebookClient(nb, timeout=BUDGET.get(rel, DEFAULT_BUDGET), km=_kernel_manager(),
                   resources={'metadata': {'path': str(dst)}}).execute(cleanup_kc=True)
    del nb.cells[0]
    nbformat.write(nb, str(run_dir/rel))
    return json.loads(nbformat.writes(nb))

def drift(rel: str, committed: dict, fresh: dict) -> tuple[list[str], list[str]]:
    "(hard, soft): prose numbers the fresh run no longer prints, split by how deterministic they are"
    was = {v.detail for v in check_r4(rel, committed)}   # already unbacked before the run
    hard, soft = [], []
    for v in check_r4(rel, fresh):
        if v.detail in was: continue
        # counts, sizes and ratios are deterministic; latencies and accuracies are not
        deterministic = bool(DRIFT_FAILS.search(v.detail)) or bool(re.fullmatch(r'-?[\d,]{3,}', v.detail))
        (hard if deterministic else soft).append(f'cell {v.where}: prose says {v.detail}')
    return hard, soft

NBS_EXEC = nbs_dir()
_pattern = os.environ.get('FASTERAI_TUTORIALS', '*')
_pages = [p for p in discover_pages(NBS_EXEC) if fnmatch.fnmatch(p, _pattern)]
_run_dir = Path(tempfile.mkdtemp(prefix='fasterai-tutorial-runs-'))
_rows, _hard = [], []
print(f'running {len(_pages)} page(s) in {_run_dir}')
for _rel in _pages:
    _t0 = time.perf_counter()
    _committed = load_page(NBS_EXEC, _rel)
    _fresh = execute_page(NBS_EXEC, _rel, _run_dir)
    _fails = check_r1(_rel, _fresh) + check_r2(_rel, _fresh) + check_r3(_rel, _fresh)
    _h, _s = drift(_rel, _committed, _fresh)
    _hard += [f'{_rel}: {x}' for x in [str(v) for v in _fails] + _h]
    _rows.append((_rel, time.perf_counter() - _t0, len(_h), len(_s)))
    for _x in _s: print(f'  DRIFT (warn) {_rel}: {_x}')
print(f'\n{"page":52s} {"seconds":>9s} {"hard":>5s} {"warn":>5s}')
for _r in _rows: print(f'{_r[0]:52s} {_r[1]:9.1f} {_r[2]:5d} {_r[3]:5d}')
test_eq(_hard, [])

In [ ]:
#| hide
# The gate. Everything above is machinery; this is the assertion CI runs.
NBS = nbs_dir()
PAGES, VIOLATIONS = run_gate(NBS)
PROBLEMS = gate_problems(PAGES, VIOLATIONS, QUARANTINE, _LANDED, set(PAGES))
LEFT = remaining(VIOLATIONS, QUARANTINE)

if PROBLEMS or LEFT:
    print(f'{len(PAGES)} pages, {len(VIOLATIONS)} violations, '
          f'{sum(len(r) for r in QUARANTINE.values())} (page, rule) pairs quarantined\n')
    for _p in PROBLEMS: print('QUARANTINE   ' + _p)
    for _page in sorted({v.page for v in LEFT}):
        _note = OPEN_AT_LANDING.get(_page)
        print(f'\n{_page}' + (f'\n  ^ known open item: {_note}' if _note else ''))
        for _v in LEFT:
            if _v.page == _page: print(f'  {_v.rule:4s} {_v.where:11s} {_v.detail[:130]}')
test_eq(PROBLEMS, [])
test_eq(LEFT, [])

---

## See Also

- [Deployable INT8 Export](tutorials/quantize/deployable_export.html) - The page these rules were written from
- [Quick Start](quickstart.html) - The shortest tour of the library
- [Overview](overview.html) - What each module does